# Phase 6.3 — recovery runner validation

This notebook checks that the Phase 6 comparison runner is safe to use with the verified Phase 6.2 recovery dataset. It does **not** train a model, compare feature performance, or make an accuracy claim. Those comparisons belong to Steps 6.4–6.8.

Before running this notebook, ensure the completed Phase 6.2 recovery files are available locally under `data/processed/model_inputs/phase6_recovery_v1/` and `reports/model_evaluation/phase6_recovery_v1/`.

In [1]:
from pathlib import Path
import subprocess
import sys

BRANCH = 'feature/model-evaluation'
COLAB_REPOSITORY = Path('/content/HalalifyAI')
if 'google.colab' in sys.modules:
    if not (COLAB_REPOSITORY / '.git').is_dir():
        subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', 'https://github.com/frager098/HalalifyAI.git', str(COLAB_REPOSITORY)], check=True)
    root = COLAB_REPOSITORY.resolve()
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(root / 'requirements.txt'), '-r', str(root / 'requirements-cleaning.txt')], check=True)
else:
    root = Path.cwd().resolve()
    while root != root.parent and not (root / 'src' / 'model_training').is_dir():
        root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from src.notebook_environment import ensure_phase6_recovery_inputs, prepare_repository
root = prepare_repository(branch=BRANCH, colab_repository=COLAB_REPOSITORY)
ensure_phase6_recovery_inputs(root)
%cd {root}
print(f'Project folder: {root}')


FileNotFoundError: Open this notebook from inside the HalalifyAI repository.

## Run the no-fit validation

This checks that the dataset, metadata, time-based protocol, and Phase 6.2 inventory still agree. It then records the 14 planned comparisons. No model is fitted.

In [ ]:
from src.model_training.feature_ablation.ablation_runner import (
    DEFAULT_RECOVERY_RUN_PLAN,
    write_recovery_runner_validation_report,
)

runner_report = write_recovery_runner_validation_report()
print(f"Status: {runner_report['status']}")
print(f"Planned comparisons: {len(runner_report['planned_experiments'])}")
print(f"Tasks: {', '.join(runner_report['tasks'])}")
print(f"Model fitting performed: {runner_report['model_fitting_performed']}")
print(f"Validation report: {DEFAULT_RECOVERY_RUN_PLAN}")
runner_report['planned_experiments']


## Check the result

A successful result says `validated`, lists 14 planned comparisons, and says `model_fitting_performed: False`. That proves the runner’s safety rules are ready. It does not tell us whether momentum, volatility, SPY, trend, or volume improves the model. Those are later controlled experiments.

The report is created at `reports/model_evaluation/phase6_recovery_v1/runner_validation.json`. It is local generated evidence and is intentionally not pushed to GitHub.